# Day 20: XGBoost Regression

Learn numerical prediction using XGBoost. Includes installation, training, evaluation, comparison with other tree models, parameter experiments, feature importance, and exercises.

Packages: pandas, numpy, matplotlib, scikit-learn, xgboost.

## 1. Install XGBoost
Run this if the package is not installed.

In [ ]:
%pip install xgboost

## 2. Import libraries and load data
The California Housing dataset has housing features and a numerical median house-value target. It may download on first use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data shape:", X.shape)
display(X.head())
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

## 3. Train XGBoost Regressor
XGBoost builds boosted trees in sequence. `n_estimators` is the number of boosting rounds, `learning_rate` controls each tree's contribution, and `max_depth` limits tree depth.

In [ ]:
model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                     objective="reg:squarederror", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

## 4. Evaluate the model

In [ ]:
pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 3))
print("R2:", round(r2_score(y_test, pred), 3))
display(pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": pred}).head(10))
plt.figure(figsize=(6,5))
plt.scatter(y_test, pred, alpha=0.25)
a,b=min(y_test.min(),pred.min()),max(y_test.max(),pred.max())
plt.plot([a,b],[a,b],"--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Actual vs Predicted")
plt.show()

## 5. Compare with Random Forest and Gradient Boosting

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
estimators = {
 "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, objective="reg:squarederror", random_state=42, n_jobs=-1),
 "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
 "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}
rows=[]
for name, est in estimators.items():
    est.fit(X_train,y_train)
    p=est.predict(X_test)
    rows.append({"Model":name,"MAE":mean_absolute_error(y_test,p),
                 "RMSE":np.sqrt(mean_squared_error(y_test,p)),"R2":r2_score(y_test,p)})
display(pd.DataFrame(rows).sort_values("RMSE"))

## 6. Experiment with max_depth

In [ ]:
for depth in [2,3,4,6,8]:
    trial=XGBRegressor(n_estimators=150,learning_rate=0.05,max_depth=depth,
                       objective="reg:squarederror",random_state=42,n_jobs=-1)
    trial.fit(X_train,y_train); p=trial.predict(X_test)
    print(f"Depth={depth} | MAE={mean_absolute_error(y_test,p):.3f} | R2={r2_score(y_test,p):.3f}")

## 7. Feature importance

In [ ]:
importance=pd.Series(model.feature_importances_,index=X.columns).sort_values(ascending=False)
display(importance.to_frame("Importance"))
importance.sort_values().plot(kind="barh",figsize=(8,5))
plt.title("XGBoost Feature Importance"); plt.show()

## 8. Practice
1. Change learning_rate to 0.1.
2. Try n_estimators=100.
3. Change max_depth to 3 and 6.
4. Find the lowest RMSE in the comparison table.
5. Print the three most important features.

**Key points:** XGBoost is a boosted-tree algorithm. It builds trees sequentially. Tune learning_rate, n_estimators, and max_depth.

Next: Day 21 — Elastic Net Regression.